# JedAI — Free Training Notebook

Train your own GPT-2-style language model **for free** on a Google Colab GPU.

**Before you start:** go to `Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU`, then `Save`.

This notebook runs the full pipeline end-to-end:
**download data -> clean -> train tokenizer -> tokenize -> train model -> generate text.**

The default uses `wikitext-2` (~2M tokens), which finishes in a few minutes so you can confirm everything works. Once it does, switch to `wikitext-103` and raise `max_steps` for a real model.

## 1. Get the code + install dependencies

In [ ]:
!git clone https://github.com/jadetindoy/jed-ai.git
%cd jed-ai
# Colab already ships a CUDA build of torch, so we only add the light extras.
!pip install -q tokenizers datasets rich pyyaml tqdm

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected -> Runtime -> Change runtime type -> T4 GPU (it will still run on CPU, just slower).')

## 2. Download real text data

`wikitext-2` for a fast first run. Change to `wikitext-103` later for a serious model.

In [ ]:
!python -m data.download_dataset --dataset wikitext-2

## 3. Clean the text

In [ ]:
!python -m data.clean --input_dir data/raw --output_dir data/cleaned

## 4. Train the tokenizer (BPE, 8k vocab)

In [ ]:
!python -m tokenizer.train --input_dir data/cleaned --vocab_size 8000

## 5. Tokenize the dataset into train/val binaries

In [ ]:
!python -m data.tokenize_dataset --input_dir data/cleaned --output_dir data/tokenized

## 6. Train the model

Uses `configs/free.yaml` (~14M params). `vocab_size` is auto-synced to the tokenizer you just trained. Bump `max_steps` in the config for a better model.

In [ ]:
!python training/trainer.py --config configs/free.yaml

## 7. Generate text from your trained model

In [ ]:
from pathlib import Path
from inference.generate import Generator

ckpts = sorted(Path('checkpoints/free').glob('ckpt_*.pt'))
assert ckpts, 'No checkpoint found - did training finish?'
print('Using checkpoint:', ckpts[-1])

gen = Generator(model_path=str(ckpts[-1]), tokenizer_path='tokenizer/tokenizer.json')
print(gen.generate('The history of', max_new_tokens=60, temperature=0.8, top_k=40))

## What next?

- The text won't be perfect — `wikitext-2` is tiny and 3000 steps is short. That's expected.
- For a real small model: rerun **step 2** with `--dataset wikitext-103`, raise `max_steps` to `20000+` in `configs/free.yaml`, and retrain.
- Save your checkpoint before the Colab session ends (it gets wiped):
  ```python
  from google.colab import files
  files.download(str(ckpts[-1]))
  ```